# Chapter 14 - RNNs, Attention, and Transformers

This notebook introduces the transition from sequential recurrent processing to attention-based models. We will keep the examples small and focus on the mechanics: scaled dot-product attention, padding and causal masks, multi-head reshaping, positional information, and a compact pre-norm transformer block.

The goal is intuition and runnable code, not a full language-model training pipeline. Exercises will live in a separate notebook later.

## Setup

We use tiny toy tensors throughout so the attention and transformer shapes stay easy to inspect.

In [ ]:
# !pip -q install torch matplotlib  # install dependencies if needed

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

plt.style.use("seaborn-v0_8")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ value, weights


def show_attention(ax, weights, title):
    image = ax.imshow(weights, vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Keys")
    ax.set_ylabel("Queries")
    return image

## From RNNs to attention

An RNN updates one hidden state at a time, so information flows through the sequence step by step. Attention replaces that recurrence with direct token-to-token interactions, which makes it easier to compare all positions at once.

In [ ]:
tokens = torch.tensor(
    [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [1.0, 1.0, 0.0, 0.0],
    ]
)
Wx = torch.tensor(
    [
        [0.8, -0.3, 0.2, 0.1],
        [0.1, 0.7, -0.4, 0.2],
        [0.2, 0.0, 0.9, -0.2],
        [0.0, 0.1, 0.2, 0.6],
    ]
)
Wh = 0.5 * torch.eye(4)

h = torch.zeros(4)
rnn_states = []
for step, x_t in enumerate(tokens, start=1):
    h = torch.tanh(x_t @ Wx + h @ Wh)
    rnn_states.append(h)
    print(f"Step {step} hidden state: {torch.round(h, decimals=3)}")

query = tokens[-1:].unsqueeze(0)
key_value = tokens.unsqueeze(0)
_, attn_weights = scaled_dot_product_attention(query, key_value, key_value)
print()
print("Attention weights from the last token to every position:")
print(torch.round(attn_weights.squeeze(0).squeeze(0), decimals=3))
print()
print("Takeaway: the RNN touches one step at a time, while attention can score all earlier positions in one operation.")

## Scaled dot-product attention

Attention compares queries with keys, rescales the similarity scores by the key dimension, and then mixes the values using the resulting weights.

In [ ]:
x = torch.tensor(
    [[[1.0, 0.0, 1.0, 0.0],
      [0.5, 1.0, 0.0, 0.5],
      [0.0, 1.0, 1.0, 0.0],
      [1.0, 0.5, 0.0, 1.0]]]
)
attention_out, attention_weights = scaled_dot_product_attention(x, x, x)

print(f"Attention output shape: {tuple(attention_out.shape)}")
print(f"Attention weight shape: {tuple(attention_weights.shape)}")
print(f"Row sums: {torch.round(attention_weights.sum(dim=-1), decimals=3)}")

fig, ax = plt.subplots(figsize=(4, 3))
show_attention(ax, attention_weights[0].detach().numpy(), "Unmasked attention")
plt.show()

## Masks: padding vs causal

A padding mask hides positions that are only padding tokens. A causal mask hides future positions so a token cannot look ahead during autoregressive decoding.

In [ ]:
batch_x = torch.randn(2, 4, 4)
lengths = torch.tensor([4, 2])
padding_mask = torch.arange(4).unsqueeze(0) < lengths.unsqueeze(1)
causal_mask = torch.tril(torch.ones(4, 4, dtype=torch.bool))
combined_mask = padding_mask[:, None, :] & causal_mask.unsqueeze(0)

_, padding_weights = scaled_dot_product_attention(batch_x, batch_x, batch_x, mask=padding_mask[:, None, :])
_, combined_weights = scaled_dot_product_attention(batch_x, batch_x, batch_x, mask=combined_mask)

print(f"Padding mask shape: {tuple(padding_mask.shape)}")
print(f"Causal mask shape: {tuple(causal_mask.shape)}")
print(f"Combined mask shape: {tuple(combined_mask.shape)}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
show_attention(axes[0], padding_weights[1].detach().numpy(), "Padding mask only")
show_attention(axes[1], combined_weights[1].detach().numpy(), "Padding + causal mask")
plt.tight_layout()
plt.show()

## Multi-head attention and shape logic

Multi-head attention splits the model dimension into smaller head dimensions, runs attention in parallel, and then combines the heads back into the original model width.

In [ ]:
batch_size, seq_len, d_model, heads = 2, 5, 8, 2
d_head = d_model // heads
x = torch.randn(batch_size, seq_len, d_model)

Wq = nn.Linear(d_model, d_model, bias=False)
Wk = nn.Linear(d_model, d_model, bias=False)
Wv = nn.Linear(d_model, d_model, bias=False)

q = Wq(x).view(batch_size, seq_len, heads, d_head).transpose(1, 2)
k = Wk(x).view(batch_size, seq_len, heads, d_head).transpose(1, 2)
v = Wv(x).view(batch_size, seq_len, heads, d_head).transpose(1, 2)

head_out, head_weights = scaled_dot_product_attention(q, k, v)
combined = head_out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

print(f"Input shape: {tuple(x.shape)}")
print(f"Query per-head shape: {tuple(q.shape)}")
print(f"Attention weight shape: {tuple(head_weights.shape)}")
print(f"Combined output shape: {tuple(combined.shape)}")

## Positional information

Attention alone is permutation-friendly, so it needs an extra signal to represent token order. Sinusoidal positional encodings are a simple way to add location-dependent structure.

In [ ]:
def sinusoidal_positions(seq_len, d_model):
    positions = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
    div_terms = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
    )
    encodings = torch.zeros(seq_len, d_model)
    encodings[:, 0::2] = torch.sin(positions * div_terms)
    encodings[:, 1::2] = torch.cos(positions * div_terms)
    return encodings


seq_len, d_model = 6, 8
token_ids = torch.tensor([2, 3, 4, 2, 5, 3])
token_embedding = nn.Embedding(6, d_model)
token_vectors = token_embedding(token_ids)
pos_vectors = sinusoidal_positions(seq_len, d_model)
combined_vectors = token_vectors + pos_vectors

print(f"Same token embedding reused at different positions: {torch.allclose(token_vectors[0], token_vectors[3])}")
print(f"Same token after adding positions: {torch.allclose(combined_vectors[0], combined_vectors[3])}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(pos_vectors[:, :4])
ax.set_title("First four sinusoidal dimensions across positions")
ax.set_xlabel("Position")
ax.set_ylabel("Value")
plt.show()

## Transformer block (pre-norm)

A pre-norm transformer block applies layer normalization before self-attention and before the feed-forward network. Residual connections wrap both sublayers.

In [ ]:
class PreNormTransformerBlock(nn.Module):
    def __init__(self, d_model=8, heads=2, mlp_ratio=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.ReLU(),
            nn.Linear(d_model * mlp_ratio, d_model),
        )

    def forward(self, x, *, key_padding_mask=None, causal_mask=None):
        h = self.norm1(x)
        attn_out, attn_weights = self.attn(
            h,
            h,
            h,
            key_padding_mask=key_padding_mask,
            attn_mask=causal_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights


block = PreNormTransformerBlock(d_model=8, heads=2)
block_input = torch.randn(2, 4, 8)
key_padding_mask = torch.tensor([
    [False, False, False, False],
    [False, False, True, True],
])
causal_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
block_output, block_weights = block(
    block_input,
    key_padding_mask=key_padding_mask,
    causal_mask=causal_mask,
)

print(f"Block input shape: {tuple(block_input.shape)}")
print(f"Block output shape: {tuple(block_output.shape)}")
print(f"Attention weight shape: {tuple(block_weights.shape)}")
print(block)

## Summary

This notebook kept Chapter 14 focused on a few transformer foundations:

- RNNs process sequences step by step, while attention compares positions directly.
- Scaled dot-product attention turns similarity scores into value-weighted mixtures.
- Padding masks and causal masks solve different problems and often need to be combined.
- Multi-head attention splits and recombines the model dimension so multiple attention patterns can run in parallel.
- Positional information gives attention-based models a notion of order.
- A pre-norm transformer block combines normalization, self-attention, residuals, and an MLP in one reusable unit.

The exercises notebook can build on this with the actual Chapter 14 end-of-chapter tasks.